In [2]:
from pathlib import Path 

import cv2
import IPython
import pandas as pd

In [3]:
def show_image(image):
    _, ret = cv2.imencode('.jpg', image)
    i = IPython.display.Image(data=ret)
    IPython.display.display(i)

In [4]:
def box_from_row(row):
    return [int(x) for x in (row['x1'], row['y1'], row['x2'], row['y2'])]

In [5]:
def draw_box(image, box):
    x1, y1, x2, y2 = box
    color = (0, 0, 255)
    thickness = 2
    cv2.rectangle(image, (x1, y1), (x2, y2), color, thickness, cv2.LINE_AA)
    show_image(image)

In [6]:
def get_by_id(id):
    faces = pd.read_csv('../data/faces.csv', index_col=0)
    fp = Path('../test_images').joinpath(faces.iloc[id]['name'])
    print(fp.name)
    img = cv2.imread(str(fp))
    return img

In [7]:
def show_by_id(id):
    img = get_by_id(id)
    show_image(img)

In [8]:
def get_center(box):
    center_x = int((box['x2'] - box['x1'])/2) + box['x1']
    center_y = int((box['y2'] - box['y1'])/2) + box['y1']
    return (center_x, center_y)

In [9]:
def check_center(row):
    center = get_center(row)
    box = row[['x1_ground', 'y1_ground', 'x2_ground', 'y2_ground']].to_frame().transpose().rename({'x1_ground': 'x1',
                                                                            'y1_ground': 'y1',
                                                                            'x2_ground': 'x2',
                                                                            'y2_ground': 'y2'},
                                                                            axis=1).to_dict(orient='list')
    center_ground = get_center({k: v[0] for k, v in box.items()})
    return center, center_ground

In [10]:
faces = pd.read_csv('../data/faces.csv', index_col=0)
faces.head()

,name,img_height,img_width,x1,y1,x2,y2,width,height,area,pct_of_frame,face_num
0,Billions.S01E02.1080p.BluRay.x265-RARBG_40128.png,1080,1920,323,5,919,731,596,726,432696,0.208669,0
1,Billions.S01E09.1080p.BluRay.x265-RARBG_52704.png,1080,1920,262,42,614,507,352,465,163680,0.078935,0
2,Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png,1080,1920,484,389,671,617,187,228,42636,0.020561,0
3,Billions.S01E10.1080p.BluRay.x265-RARBG_51504.png,1080,1920,1003,42,1195,261,192,219,42048,0.020278,1
4,Billions.S01E12.1080p.BluRay.x265-RARBG_78336.png,1080,1920,1121,66,1662,824,541,758,410078,0.197761,0


In [11]:
d = Path('../test_scripts/results')
d.exists()

False

In [12]:
files = [x for x in d.iterdir() if x.suffix == '.csv']
dfs = [pd.read_csv(str(x), index_col=0) for x in files]
df = pd.concat(dfs)
df = df.drop('index', axis=1)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../test_scripts/results'

In [4]:
retina = df[df['detector'] == 'retina_accuracy.py']
retina

,index,x1,y1,x2,y2,width,height,area,confidence,img_width,...,x2_ground,y2_ground,x1_diff,y1_diff,x2_diff,y2_diff,area_diff,pct_diff,id,detector
6,0,190.0,670.0,295.0,794.0,105.0,124.0,13020.0,0.999489,1920,...,420,896,116.0,-364.0,11.0,-488.0,9666.0,0.004661,1494,retina_accuracy.py
7,1,1188.0,241.0,1329.0,430.0,141.0,189.0,26649.0,0.999233,1920,...,420,896,-882.0,65.0,-1023.0,-124.0,-3963.0,-0.001911,1494,retina_accuracy.py
8,2,304.0,735.0,390.0,885.0,86.0,150.0,12900.0,0.999225,1920,...,420,896,2.0,-429.0,-84.0,-579.0,9786.0,0.004719,1494,retina_accuracy.py
9,3,748.0,445.0,840.0,568.0,92.0,123.0,11316.0,0.998956,1920,...,420,896,-442.0,-139.0,-534.0,-262.0,11370.0,0.005483,1494,retina_accuracy.py
3,0,622.0,292.0,663.0,366.0,41.0,74.0,3034.0,0.993857,1920,...,1305,328,648.0,978.0,607.0,904.0,-1494.0,-0.000720,1346,retina_accuracy.py
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3,0,1285.0,282.0,1631.0,764.0,346.0,482.0,166772.0,0.995858,1920,...,1635,776,-76.0,927.0,-422.0,445.0,49210.0,0.023732,1660,retina_accuracy.py
3,0,1049.0,80.0,1351.0,520.0,302.0,440.0,132880.0,0.997892,1920,...,250,839,-956.0,13.0,-1258.0,-427.0,-99596.0,-0.048030,36,retina_accuracy.py
4,1,119.0,655.0,238.0,822.0,119.0,167.0,19873.0,0.997611,1920,...,250,839,-26.0,-562.0,-145.0,-729.0,13411.0,0.006467,36,retina_accuracy.py
3,0,257.0,58.0,360.0,183.0,103.0,125.0,12875.0,0.999611,640,...,365,191,-28.0,171.0,-131.0,46.0,6437.0,0.020954,1795,retina_accuracy.py


In [23]:
row = df.iloc[0]
box = row[['x1_ground', 'y1_ground', 'x2_ground', 'y2_ground']].to_frame().transpose().rename({'x1_ground': 'x1',
                                                                            'y1_ground': 'y1',
                                                                            'x2_ground': 'x2',
                                                                            'y2_ground': 'y2'},
                                                                            axis=1).to_dict(orient='list')

In [19]:
{k: v for k, v in box.items()}

{'x1': [306], 'y1': [697], 'x2': [420], 'y2': [896]}

In [24]:
box

{'x1': [306], 'y1': [697], 'x2': [420], 'y2': [896]}

In [29]:
check_center(row)

((1252.0, 333.0), (363, 796))

In [48]:
draw_box(get_by_id(row['id']), box_from_row(row))

The.X-Files.S01E19.Shapes.1080p.BluRay.10Bit.Dts-HDMa5.1.HEVC-d3g_11640.png


error: OpenCV(4.9.0) :-1: error: (-5:Bad argument) in function 'rectangle'
> Overload resolution failed:
>  - Can't parse 'pt1'. Sequence item with index 0 has a wrong type
>  - Can't parse 'pt1'. Sequence item with index 0 has a wrong type
>  - Can't parse 'rec'. Expected sequence length 4, got 2
>  - Can't parse 'rec'. Expected sequence length 4, got 2


In [36]:
str(Path('../test_images').joinpath(row['name']))

'../test_images/The.X-Files.S01E19.Shapes.1080p.BluRay.10Bit.Dts-HDMa5.1.HEVC-d3g_11640.png'

In [37]:
Path('../test_images').joinpath(row['name']).exists()

True

In [49]:
box_from_row(row)

(1170.0, 251.0, 1334.0, 415.0)